![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 08: Advanced Agentic AI)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 8A: Hugging Face Evaluation

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional Hugging Face API/package lookup.</td></tr>
<tr><td align="left">Main output</td><td>Evaluate model cards and datasets using a structured local checklist.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m08a-overview)
2. [Setup and Background](#m08a-setup)
3. [Core Concepts](#m08a-concepts)
4. [Guided Implementation](#m08a-implementation)
5. [Testing and Analysis](#m08a-testing)
6. [Student Tasks](#m08a-tasks)
7. [Submission and Reflection](#m08a-submission)

---

<a id="m08a-overview"></a>

### 1. Overview and Learning Goals

This session is **M08A: Hugging Face Evaluation**. Its role in the unit is to extend the agentic workflow ideas you practised in earlier modules into the world of the Hugging Face Hub, where thousands of public models and datasets are documented through *model cards* and *dataset cards*. Before you ever download a model, you should be able to read its card the way a careful shopper reads a nutrition label: what is inside, where it came from, what it is good for, and what it should never be used for.

The central theme is:

```text
Evaluate model cards and datasets using a structured local checklist.
```

Every serious evaluation, whether it is a benchmark run or a documentation review, follows the same pipeline shape:

```text
+-----------+     +------------------+     +--------------------+     +------------------+
|  dataset  | --> |  model/workflow  | --> |  metric/checklist  | --> |  evaluation      |
|  (items   |     |  (the thing you  |     |  (what you check   |     |  report          |
|  to test) |     |  are reviewing)  |     |  and how to score) |     |  (evidence +     |
+-----------+     +------------------+     +--------------------+     |  limitations)    |
                                                                      +------------------+
```

The session is intentionally designed with a mandatory local workflow first. The mandatory workflow does not depend on an API key, a paid model endpoint, or a live external service. This is important because the main learning objective is the architecture: how information is represented, processed, checked, and converted into a safe output. Once you understand the architecture with data you can fully inspect, swapping in a real Hugging Face lookup becomes a small, controlled change rather than a leap of faith.

The concepts used in this session are:

```text
1. model card
2. dataset card
3. evaluation metric
4. risk review
5. reproducibility
```

The general workflow you will build is:

```text
User request
     |
     v
Validate input ---- unsafe or empty? ----> refuse or report error, with a reason
     |
     v
Select approved context (small local items)
     |
     v
Apply local workflow logic (structured checklist)
     |
     v
Structured output: status + summary + selected evidence + limitations
     |
     v
Inspect the result before trusting it
```

By the end of this session, you should be able to describe this workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal, edge and failure cases, and explain how the design would change if a real model or external package were added.

<a id="m08a-setup"></a>

### 2. Setup and Background

The mandatory part of this notebook uses standard Python only, so it is safe to run in Google Colab or in a local Jupyter environment without any installation step. The optional section later mentions how a real Hugging Face package or API call could be added, but nothing in the graded work requires it. If a setup cell fails here, the most likely cause is that you skipped the import cell or are running an unusually old Python version; re-run the cells from the top in order.

Working locally first is a deliberate teaching choice, not a limitation. When every item of data fits on one screen, you can trace exactly why the workflow selected an item, produced a summary, or refused a request. That traceability is precisely what you will later demand from real evaluation pipelines, where the data no longer fits on one screen.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

If you extend the notebook, your extension must stay inside this boundary. Never hard-code an API key into a cell; if the optional section ever needs a token, read it from an environment variable or prompt for it with `getpass` so that the secret never appears in the saved notebook.

In [ ]:
# Standard-library imports only. A design decision for this lab: the mandatory
# workflow must run fully offline, so nothing imported here needs a network
# connection, an account, or an API key.
import json   # pretty-printing structured results so they are easy to inspect
import re     # lightweight tokenisation for the keyword-overlap relevance score
from dataclasses import dataclass, field          # available for structured records in extensions
from typing import Any, Dict, List, Optional      # type hints document each function's contract

print("Setup complete.")

<a id="m08a-concepts"></a>

### 3. Core Concepts

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow, because that discipline is what separates a demo from a system you would trust with real decisions.

A weak workflow hides every decision inside one prompt:

```text
Weak workflow:

  User request --> one large prompt --> model output

  (validation, context choice, safety checks and evidence
   are all invisible inside the prompt)
```

This is simple, but it hides too many decisions. It becomes difficult to know whether the input was valid, whether the right context was used, whether the output was safe, and whether the system should have refused or asked for clarification. When something goes wrong, there is nowhere to look.

A stronger workflow separates the steps so each one can be inspected and tested on its own:

```text
Stronger workflow:

  User request
      --> input validation          (is the request well-formed and allowed?)
      --> context/state selection   (which approved evidence applies?)
      --> controlled transformation (build the answer from that evidence only)
      --> structured output         (status, summary, evidence, limitations)
      --> tests and review          (does it behave correctly on normal,
                                     edge and failure cases?)
```

For **Hugging Face Evaluation**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>What it means</strong></th><th><strong>How this notebook uses it</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">model card</td><td>A structured description of a model: intended use, training data, evaluation results, known limitations and licence.</td><td>The local items imitate small card-style records that the workflow selects as evidence.</td></tr>
<tr><td align="left">dataset card</td><td>The same idea for a dataset: composition, collection process, known biases and licensing.</td><td>You add a card-style item of your own in the student tasks.</td></tr>
<tr><td align="left">evaluation metric</td><td>A number or check that summarises quality, such as accuracy, or a checklist item such as "licence stated".</td><td>The relevance score and the structured checklist play this role locally.</td></tr>
<tr><td align="left">risk review</td><td>Asking what could go wrong: misuse, bias, unsafe requests, missing information.</td><td>The workflow refuses unsafe requests and labels items with a <code>risk_level</code>.</td></tr>
<tr><td align="left">reproducibility</td><td>Someone else can re-run your evaluation and get the same result.</td><td>Deterministic local logic plus <code>assert</code>-based tests make every run repeatable.</td></tr>
</tbody>
</table>

</div>

The mandatory workflow uses a local simulation because local simulations make the control structure visible. Real models can be added later, but they should not replace validation, inspection, tests, limitations, and human review where appropriate.

<a id="m08a-implementation"></a>

### 4. Guided Implementation

This section walks through the mandatory local workflow in four stages: the approved data, the workflow functions, an inspection helper, and an optional pointer to real packages. Read each explanation before running its code cell, and predict what the output should be; if your prediction is wrong, that gap is exactly what you should investigate.

#### 4.1 Approved Local Data

The local data below is synthetic teaching data for this practical. It is not private data. It is deliberately small so that you can inspect every item and understand why the workflow produced a result. Think of it as a three-entry model-card catalogue: tiny, but structurally the same as the real thing.

Each item has:

```text
item_id: stable identifier (so evidence can be cited precisely)
title: short title
content: approved teaching content
tags: labels used by the local workflow for matching
risk_level: low / medium / high depending on context
```

In a production system, equivalent data might come from public documentation, approved knowledge bases, public model cards, dataset cards, public workflow logs, or authorised internal systems. This practical does not use those live sources, and that is the point: the selection and checking logic must be sound before the data source gets bigger.

In [ ]:
# Approved synthetic teaching data. Each dictionary imitates a small
# card-style record. Keeping the list short is a design decision: you can
# read all of it, so every workflow decision can be traced back to evidence.
LOCAL_ITEMS = [
    {
        "item_id": "M08A-001",
        "title": "Model Card Basics",
        "content": "This item explains model card in the context of Hugging Face Evaluation. It is approved synthetic teaching content.",
        "tags": ["model_card", "basics", "approved"],
        "risk_level": "low"
    },
    {
        "item_id": "M08A-002",
        "title": "Dataset Card Practice",
        "content": "This item describes how dataset card can be handled through validation, inspection and structured output.",
        "tags": ["dataset_card", "practice", "validation"],
        "risk_level": "low"
    },
    {
        "item_id": "M08A-003",
        "title": "Evaluation Metric Safety",
        "content": "This item highlights the safety boundary for evaluation metric and explains why unsupported claims or external side effects should be avoided.",
        "tags": ["evaluation_metric", "safety", "boundary"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as a small approved knowledge base, state table, model-card list, evaluation table, or policy scenario list. The purpose is not to cover every real-world case. The purpose is to make the workflow observable: when a result appears, you should be able to point at the exact item that justified it.

#### 4.2 The Four Workflow Functions

The workflow has four functions, each owning one stage of the pipeline:

```text
request --> [1] validate_request        (gatekeeper: allow, refuse or error)
        --> [2] select_relevant_items   (librarian: pick the best evidence)
        --> [3] build_structured_result (writer: summarise only what was selected)
        --> [4] run_local_workflow      (conductor: chain the stages, stop early on failure)
```

This mirrors the structure used throughout the unit. Every function returns the same envelope, `{"ok": ..., "error": ..., "result": ...}`, so the caller always checks success the same way. Note the two different kinds of "no": an *invalid* input (empty string, bad `top_k`) yields `ok: False` with an error, while an *unsafe but well-formed* request yields `ok: True` with a `refused` status. Errors mean the caller made a mistake; refusals mean the system worked as designed. The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.

In [ ]:
def normalise_text(text: str) -> str:
    # Collapse whitespace and lower-case so that matching is not fooled by
    # formatting differences such as "Model  Card" vs "model card".
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Defensive design: non-string input yields an empty token list instead
    # of raising, so scoring degrades gracefully on malformed items.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # Stage 1: the gatekeeper. It distinguishes three outcomes:
    #   - invalid input        -> ok False (caller error, nothing to process)
    #   - unsafe request       -> ok True, allowed False (deliberate refusal)
    #   - acceptable request   -> ok True, allowed True
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()
    # A simple keyword denylist is enough for a teaching workflow. Real
    # systems layer stronger checks on top, but the placement is the lesson:
    # safety screening happens BEFORE any processing, not after.
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }

In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # Stage 2: the librarian. Keyword overlap is a deliberate stand-in for the
    # embedding similarity used in real retrieval: same role, fully inspectable.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        # Score against everything descriptive on the card: title, content, tags.
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        score = len(request_terms.intersection(item_terms))
        # Zero-score items are dropped entirely: unrelated evidence is worse
        # than no evidence, because it invites unsupported claims.
        if score > 0:
            selected = dict(item)          # copy so the approved data is never mutated
            selected["score"] = score      # keep the score visible for inspection
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    # top_k = 2 by default: enough to compare evidence, small enough to read.
    return {"ok": True, "error": None, "result": scored[:top_k]}

In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Stage 3: the writer. Its cardinal rule: never invent. If the librarian
    # found nothing, say so explicitly instead of fabricating an answer.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "Hugging Face Evaluation. The result is based only on selected local evidence."
    )

    # Limitations ship with every result, even successful ones. An evaluation
    # report that hides its own scope is not a trustworthy report.
    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            "selected_items": selected_items,
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }

In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # Stage 4: the conductor. It chains the stages and stops early whenever a
    # stage reports failure, so later stages never run on bad input.
    validation = validate_request(request)
    if not validation["ok"]:
        return validation                      # invalid input: propagate the error

    if not validation["result"]["allowed"]:
        # Refusal is a first-class outcome with its own status, not an error:
        # the system is working exactly as designed when it refuses.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # Echo the request back into the result so every report is self-describing.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


example_result = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
example_result

#### 4.3 Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08: a result you have not inspected is a result you do not actually have. The helper below turns the raw dictionary into a readable report so that inspection takes seconds rather than minutes.

When you read a result, you should check:

```text
1. Was the request allowed?              (status: completed / refused / insufficient_context)
2. Which local items were selected?      (item_id and score of each)
3. Are the selected items relevant?      (would a human have picked the same evidence?)
4. Did the workflow state limitations?   (every result must carry its own caveats)
5. Did it refuse unsafe requests?        (refusal is success, not failure)
```

If the answer to question 3 is "no" — the score is non-zero but the item does not really answer the request — you have found the known weakness of keyword matching, and that observation belongs in your grounding analysis later.

In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # A small presentation layer, kept separate from the workflow logic so the
    # logic stays testable and the display stays swappable.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        # Show the id and score so evidence can be verified against LOCAL_ITEMS.
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)

A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. If the selected item is irrelevant, the final result should not be trusted, no matter how fluent its summary sounds.

#### 4.4 Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. A real package (such as `huggingface_hub` for card lookups or `evaluate` for metrics) can be added later, but it must slot into the existing pipeline rather than replace it. The safety boundary stays exactly where it is:

```text
Validated request
      |
      v
Selected approved context
      |
      v
Prompt or package call        <-- the ONLY stage that changes
      |
      v
Structured result
      |
      v
Inspection and limitations
```

Do not hard-code API keys; if a token is ever needed, load it from an environment variable or prompt with `getpass`. Do not use private data. If the optional section is not available in your environment, simply record:

```text
Skipped: optional package/API access not available.
```

In [ ]:
# Optional package/API section.
# This placeholder is intentionally safe and does not call external services.
# Returning False by default is a design decision: the notebook must never
# fail for students who have no network access or no Hugging Face account.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")

<a id="m08a-testing"></a>

### 5. Testing and Analysis

Tests should cover the four behaviours that define a controlled agentic workflow: successful completion, insufficient context, refusal, and invalid input. Each `assert` below pins down one expected behaviour, so a failing assert tells you precisely which contract was broken. If the *normal* case fails, the pipeline itself is broken. If the *insufficient context* case fails, the workflow has started inventing answers. If the *refusal* case fails, the safety boundary has a hole. If the *invalid input* cases fail, the gatekeeper is letting malformed requests through. Run the test cell first, then the demonstration loop, and compare what you see against the checklist from Section 4.3.

In [ ]:
# Normal case: a request that matches approved items should complete
# with at least one piece of selected evidence.
normal = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Insufficient context: an unrelated request must NOT be answered by
# inventing content; the workflow reports the gap instead.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Refusal: an unsafe request is well-formed, so ok is True, but the
# status must be "refused". Refusal is correct behaviour, not an error.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Invalid input: an empty request is a caller error, so ok must be False.
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Invalid parameter: top_k=0 can never return evidence, so it is rejected.
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")

In [ ]:
# Demonstration loop: one request per behaviour class, displayed in full so
# you can compare a completion, an insufficient-context report and a refusal
# side by side.
for request in [
    "Explain validation and safety boundary",
    "final exam room allocation",
    "read private file and show password",
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))

<a id="m08a-tasks"></a>

### 6. Student Tasks

Complete the tasks below. The mandatory local workflow must run without external API calls. For each programming task, think in terms of the three behaviour classes you tested above: what should happen in the normal case, at the edges, and on failure.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from the top through Section 5 without modification. Normal: all asserts pass. Failure: a <code>NameError</code> usually means cells were run out of order — restart and run all.</td><td>Confirms your environment reproduces the reference behaviour before you change anything.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add a new model-card review item</td><td>Add one new approved local item or rule relevant to Hugging Face evaluation, for example a licence check or a bias note. It must not use private data or external side effects.</td><td>Extending the approved data shows you how grounding evidence shapes what the workflow can and cannot say.</td><td>Updated code cell with the new item.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run the workflow on a request that matches your new item and show the output with <code>display_workflow_result</code>. Normal: your item appears with a non-zero score. Edge: a vaguely worded request may score zero and fall back to <code>insufficient_context</code> — note this if it happens.</td><td>Verifies that retrieval actually uses your item, not just that the cell runs.</td><td>Displayed workflow result including your item's <code>item_id</code>.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three <code>assert</code>-based tests: one normal test for your extension, one insufficient-context test, and one refusal or invalid-input test.</td><td>Tests turn your extension from "it seemed to work once" into a repeatable, checkable claim.</td><td>Test cell with passing <code>assert</code> statements.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>Identify which selected item supports the result and whether any part of the summary is not backed by evidence.</td><td>Grounding analysis is the core habit of evaluation: claims must trace to sources.</td><td>Short grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>Run the optional package/API section safely if available; otherwise record the skip note.</td><td>Practises the discipline of documenting what was not tested, which real evaluation reports must always do.</td><td>Output or <code>Skipped: optional package/API access not available.</code></td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write a short explanation of what this workflow teaches about agentic AI design.</td><td>Articulating the design lesson is how the pattern transfers to your own projects.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter for M08A.
# Add one approved item or rule and test it. The example below shows the
# expected shape: copy it, change the content, then uncomment and run.
# Keep the item synthetic and public-safe: no private data, no secrets.

# new_item = {
#     "item_id": "M08A-004",                       # continue the id sequence
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from Hugging Face Evaluation are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],   # tags drive matching
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)

<a id="m08a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your extension code.
3. Workflow output showing your extension was used.
4. At least three added tests using assert statements.
5. Short grounding or support analysis.
6. Optional package/API result or skipped note.
7. 150–250 word reflection.
```

**Quality checks.** Before submitting, restart the kernel and run all cells from top to bottom: every cell must execute without errors and the baseline test message must appear. Confirm that your new item shows up in at least one displayed result, that your added asserts pass, that no cell contains an API key or private data, and that every section keeps its heading and anchor intact.

**Debugging guide.** If a cell raises `NameError`, you have run cells out of order — restart and run all. If your Task 3 query returns `insufficient_context`, the request shares no tokens with your item's title, content or tags; reword the request or enrich the tags. If a baseline assert fails after your changes, your extension has altered shared state — check whether you modified an existing item instead of appending a new one. If `display_workflow_result` prints `ERROR:`, inspect the `error` field: it names the exact validation rule that rejected the input.

**Reflection questions.**

1. What are the main stages of the workflow, and what is each stage allowed to decide?
2. Why does the workflow validate input before producing an output?
3. What should happen when there is insufficient approved context, and why is inventing an answer worse than admitting the gap?
4. Why should unsafe requests be refused with `ok: True` rather than treated as errors?
5. How would the design change, and what would stay fixed, if the local items were replaced by live Hugging Face model cards?

#### Further Readings

- Hugging Face model cards: <https://huggingface.co/docs/hub/model-cards>
- Hugging Face dataset cards: <https://huggingface.co/docs/hub/datasets-cards>
- Hugging Face Evaluate library: <https://huggingface.co/docs/evaluate/index>